# Community detection on the creator crossover network -- Leiden

**Purpose**: `creator_crossover_community_detection.ipynb` ran Louvain and checked robustness to raw-vs-rank edge weighting (Adjusted Rand Index 0.253 -- poor). This notebook checks the other axis: does the **algorithm itself** matter, holding the graph and weighting fixed? Same 23-title, `enrichment_ratio`-weighted graph, run through Leiden instead of Louvain, then compared directly against Louvain on the identical graph -- not just re-reported from the other notebook, recomputed here so the comparison is apples-to-apples within one run.

**Method**: `leidenalg` + `python-igraph` (neither was previously installed; added below and to `requirements.txt`, same analysis-only, no-collector-risk reasoning as `networkx`). `leidenalg.ModularityVertexPartition` is used specifically, not Leiden's default CPM objective -- CPM and modularity optimize genuinely different things, and the only fair comparison against Louvain (which optimizes modularity) is Leiden's modularity variant. `n_iterations=-1` (run to convergence, the library's own recommendation over a fixed iteration count) and a fixed `seed` for reproducibility, same discipline as the Louvain notebook.

**Same caveat as before, still true here**: most pairwise `enrichment_ratio`s are at or below the chance baseline (see the other notebook for the exact count). Whatever Leiden finds is still relative avoidance for most of the graph, not positive affinity, except for the same handful of genuinely enriched pairs.

In [1]:
# Imports and repo path setup -- same pattern as every other notebook.
import sys
from itertools import combinations
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").is_file():
            return candidate
    raise RuntimeError("Could not find repo root (CLAUDE.md not found in any parent directory) -- run this notebook from somewhere inside the repo")

REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import yaml
import networkx as nx
import igraph as ig
import leidenalg
from scipy.stats import rankdata

from etl.db import get_connection

In [2]:
# All 23 active titles, and their genre/platform/publisher -- identical to
# creator_crossover_community_detection.ipynb.
with open(REPO_ROOT / "config" / "titles.yaml") as f:
    titles_config = yaml.safe_load(f)["titles"]

active_titles = [t for t in titles_config if t.get("is_active")]
display_name_by_id = {t["id"]: t["display_name"] for t in active_titles}
genre_by_id = {t["id"]: t.get("genre") for t in active_titles}
platform_by_id = {t["id"]: t.get("platform") for t in active_titles}
publisher_by_id = {t["id"]: t.get("publisher") for t in active_titles}
all_title_ids = sorted(display_name_by_id.keys())

print(f"{len(all_title_ids)} active titles")

23 active titles


In [3]:
# Enrichment ratio, all 253 possible pairs -- identical method and query to
# creator_crossover_community_detection.ipynb, recomputed fresh here so
# this notebook stays independently runnable (no creator_crossover_history
# table exists to read a persisted "most recent run" from either).
conn = get_connection()
raw_rows = conn.execute(
    "SELECT channel_id, title_id, viewer_count FROM viewership_snapshots WHERE is_official_broadcast = 0"
).fetchall()
conn.close()

raw_df = pd.DataFrame(raw_rows, columns=["channel_id", "title_id", "viewer_count"])
channel_title_df = raw_df.groupby(["channel_id", "title_id"])["viewer_count"].sum().rename("viewer_time").reset_index()

titles_per_channel = channel_title_df.groupby("channel_id")["title_id"].nunique()
crossover_channel_ids = titles_per_channel[titles_per_channel > 1].index
crossover_df = channel_title_df[channel_title_df["channel_id"].isin(crossover_channel_ids)]

n_total_channels = channel_title_df["channel_id"].nunique()
total_channels_by_title = channel_title_df.groupby("title_id")["channel_id"].nunique()

titles_by_channel = crossover_df.groupby("channel_id")["title_id"].apply(set)
pair_channels: dict[tuple[str, str], set] = {}
for channel_id, title_set in titles_by_channel.items():
    for a, b in combinations(sorted(title_set), 2):
        pair_channels.setdefault((a, b), set()).add(channel_id)

pair_rows = []
for a, b in combinations(all_title_ids, 2):
    shared = len(pair_channels.get((a, b), set()))
    total_a, total_b = total_channels_by_title.get(a, 0), total_channels_by_title.get(b, 0)
    expected = total_a * total_b / n_total_channels if n_total_channels else float("nan")
    enrichment = shared / expected if expected > 0 else float("nan")
    pair_rows.append({"a": a, "b": b, "shared": shared, "enrichment": enrichment})

pair_df = pd.DataFrame(pair_rows)
print(f"{n_total_channels} distinct above-threshold, non-official channels across all 23 titles")
print(f"{(pair_df['shared'] > 0).sum()} of {len(pair_df)} possible pairs have >=1 shared crossover channel")

296583 distinct above-threshold, non-official channels across all 23 titles
209 of 253 possible pairs have >=1 shared crossover channel


In [4]:
# Build the weighted graph -- identical construction to the Louvain
# notebook (nodes = 23 titles, edges = enrichment_ratio, zero-shared pairs
# get no edge at all, not a zero-weight one).
G_raw = nx.Graph()
G_raw.add_nodes_from(all_title_ids)
edge_pairs = []
for _, row in pair_df[pair_df["shared"] > 0].iterrows():
    G_raw.add_edge(row["a"], row["b"], weight=row["enrichment"])
    edge_pairs.append((row["a"], row["b"], row["enrichment"]))

print(f"Graph: {G_raw.number_of_nodes()} nodes, {G_raw.number_of_edges()} edges")

Graph: 23 nodes, 209 edges


In [5]:
# Convert to igraph (leidenalg operates on igraph.Graph, not networkx
# directly) and run Leiden with the modularity objective specifically --
# NOT leidenalg's default CPM objective, since CPM and modularity optimize
# different things and Louvain (the other notebook) optimizes modularity.
# This is the only fair, apples-to-apples comparison between the two
# algorithms.
#
# A real gotcha, caught by cross-checking rather than assumed correct:
# leidenalg's own `partition.modularity` attribute reports the UNWEIGHTED
# modularity of the found partition, not the weighted one it actually
# optimized -- confirmed directly (igraph's own `Graph.modularity(...,
# weights=...)` on the identical partition gives a completely different,
# correct-looking value; networkx's `modularity()` on the same partition
# matches that weighted value exactly, not `partition.modularity`). This
# is true even though `weights="weight"` is passed correctly to
# `find_partition` -- the partition-finding step honors the weights (the
# resulting partition is the correctly weighted optimum), the `.modularity`
# convenience attribute just doesn't report it. Reported modularity below
# is computed via `Gig.modularity(membership, weights=...)`, not the
# attribute.
LEIDEN_SEED = 42

def run_leiden(G: nx.Graph) -> tuple[list[set], float]:
    Gig = ig.Graph.from_networkx(G)
    partition = leidenalg.find_partition(
        Gig, leidenalg.ModularityVertexPartition,
        weights="weight", seed=LEIDEN_SEED, n_iterations=-1,  # -1 = run to convergence, leidenalg's own recommendation
    )
    node_names = Gig.vs["_nx_name"]
    communities = [set(node_names[i] for i in comm) for comm in partition]
    weighted_modularity = Gig.modularity(partition.membership, weights=Gig.es["weight"])
    return communities, weighted_modularity

communities_leiden_raw, modularity_leiden_raw = run_leiden(G_raw)
communities_leiden_raw = sorted(communities_leiden_raw, key=len, reverse=True)

print(f"{len(communities_leiden_raw)} communities found (Leiden, raw enrichment weights):\n")
for i, comm in enumerate(communities_leiden_raw):
    names = sorted(display_name_by_id[t] for t in comm)
    print(f"Community {i} ({len(comm)} titles): {', '.join(names)}")
print(f"\nModularity (weighted, corrected): {modularity_leiden_raw:.3f}")

4 communities found (Leiden, raw enrichment weights):

Community 0 (8 titles): Apex Legends, Fortnite, League of Legends, Overwatch, Rainbow Six Siege, Rocket League, Teamfight Tactics, VALORANT
Community 1 (6 titles): Age of Empires II, Counter-Strike 2, Dota 2, Hearthstone, PUBG: BATTLEGROUNDS, StarCraft II
Community 2 (5 titles): Brawl Stars, Free Fire, League of Legends: Wild Rift, Mobile Legends: Bang Bang, PUBG Mobile
Community 3 (4 titles): Guilty Gear -Strive-, Mortal Kombat 1, Street Fighter 6, Tekken 8

Modularity (weighted, corrected): 0.439


In [6]:
# Same edge set, reweighted by rank -- identical robustness check to the
# Louvain notebook, this time on Leiden.
weights_only = [w for _, _, w in edge_pairs]
ranks = rankdata(weights_only)

G_rank = nx.Graph()
G_rank.add_nodes_from(all_title_ids)
for (a, b, _), r in zip(edge_pairs, ranks):
    G_rank.add_edge(a, b, weight=float(r))

communities_leiden_rank, modularity_leiden_rank = run_leiden(G_rank)
communities_leiden_rank = sorted(communities_leiden_rank, key=len, reverse=True)

print(f"{len(communities_leiden_rank)} communities found (Leiden, rank-based weights):\n")
for i, comm in enumerate(communities_leiden_rank):
    names = sorted(display_name_by_id[t] for t in comm)
    print(f"Community {i} ({len(comm)} titles): {', '.join(names)}")
print(f"\nModularity: {modularity_leiden_rank:.3f}")

3 communities found (Leiden, rank-based weights):

Community 0 (10 titles): Apex Legends, Fortnite, Guilty Gear -Strive-, Mortal Kombat 1, Overwatch, Rainbow Six Siege, Rocket League, Street Fighter 6, Tekken 8, VALORANT
Community 1 (8 titles): Age of Empires II, Counter-Strike 2, Dota 2, Hearthstone, League of Legends, PUBG: BATTLEGROUNDS, StarCraft II, Teamfight Tactics
Community 2 (5 titles): Brawl Stars, Free Fire, League of Legends: Wild Rift, Mobile Legends: Bang Bang, PUBG Mobile

Modularity: 0.145


In [7]:
# Also run Louvain on this SAME graph (not re-reported from the other
# notebook -- recomputed here so the Leiden-vs-Louvain comparison below is
# on identical data from the same run, not two runs that could in
# principle have drawn on different database states).
communities_louvain_raw = nx.algorithms.community.louvain_communities(G_raw, weight="weight", seed=LEIDEN_SEED)
communities_louvain_raw = sorted(communities_louvain_raw, key=len, reverse=True)
modularity_louvain_raw = nx.algorithms.community.modularity(G_raw, communities_louvain_raw, weight="weight")

print(f"{len(communities_louvain_raw)} communities found (Louvain, raw enrichment weights, this run):\n")
for i, comm in enumerate(communities_louvain_raw):
    names = sorted(display_name_by_id[t] for t in comm)
    print(f"Community {i} ({len(comm)} titles): {', '.join(names)}")
print(f"\nModularity: {modularity_louvain_raw:.3f}")

4 communities found (Louvain, raw enrichment weights, this run):

Community 0 (8 titles): Apex Legends, Fortnite, League of Legends, Overwatch, Rainbow Six Siege, Rocket League, Teamfight Tactics, VALORANT
Community 1 (6 titles): Age of Empires II, Counter-Strike 2, Dota 2, Hearthstone, PUBG: BATTLEGROUNDS, StarCraft II
Community 2 (5 titles): Brawl Stars, Free Fire, League of Legends: Wild Rift, Mobile Legends: Bang Bang, PUBG Mobile
Community 3 (4 titles): Guilty Gear -Strive-, Mortal Kombat 1, Street Fighter 6, Tekken 8

Modularity: 0.439


In [8]:
# Adjusted Rand Index, computed by hand (see creator_crossover_community_
# detection.ipynb for why this isn't just imported from scikit-learn) --
# two comparisons: Leiden raw vs. Leiden rank (weight-scheme robustness,
# same check as the other notebook, this time for Leiden), and Leiden raw
# vs. Louvain raw (algorithm-choice robustness, the actual point of this
# notebook).
from math import comb

def labels_from_communities(communities, node_order):
    label_by_node = {}
    for i, comm in enumerate(communities):
        for node in comm:
            label_by_node[node] = i
    return [label_by_node[n] for n in node_order]

def adjusted_rand_index(labels_a, labels_b):
    df = pd.DataFrame({"a": labels_a, "b": labels_b})
    contingency = pd.crosstab(df["a"], df["b"]).values
    n = contingency.sum()
    sum_comb_c = sum(comb(int(x), 2) for x in contingency.sum(axis=1))
    sum_comb_k = sum(comb(int(x), 2) for x in contingency.sum(axis=0))
    sum_comb = sum(comb(int(x), 2) for x in contingency.flatten())
    expected = sum_comb_c * sum_comb_k / comb(int(n), 2)
    max_index = 0.5 * (sum_comb_c + sum_comb_k)
    if max_index == expected:
        return 1.0
    return (sum_comb - expected) / (max_index - expected)

labels_leiden_raw = labels_from_communities(communities_leiden_raw, all_title_ids)
labels_leiden_rank = labels_from_communities(communities_leiden_rank, all_title_ids)
labels_louvain_raw = labels_from_communities(communities_louvain_raw, all_title_ids)

ari_leiden_weight_robustness = adjusted_rand_index(labels_leiden_raw, labels_leiden_rank)
ari_algorithm_robustness = adjusted_rand_index(labels_leiden_raw, labels_louvain_raw)

print(f"ARI, Leiden raw vs. Leiden rank (weight-scheme robustness):   {ari_leiden_weight_robustness:.3f}")
print(f"ARI, Leiden raw vs. Louvain raw (algorithm-choice robustness): {ari_algorithm_robustness:.3f}")
print("(1.0 = identical partitions, 0.0 = no more agreement than chance)")

ARI, Leiden raw vs. Leiden rank (weight-scheme robustness):   0.535
ARI, Leiden raw vs. Louvain raw (algorithm-choice robustness): 1.000
(1.0 = identical partitions, 0.0 = no more agreement than chance)


In [9]:
# Audience-region profile per title -- identical derivation to
# creator_crossover_community_detection.ipynb's updated (2026-09-20)
# version: language_mix_snapshots (audience-side, viewer language mix),
# not tournaments.country (which turned out dominated by grassroots
# tournament volume, not audience location -- see that notebook for the
# full reasoning behind the swap).
LANGUAGE_TO_REGION = {
    "ko": "Asia-Pacific", "ja": "Asia-Pacific", "th": "Asia-Pacific", "vi": "Asia-Pacific",
    "id": "Asia-Pacific", "tl": "Asia-Pacific", "hi": "Asia-Pacific", "ms": "Asia-Pacific",
    "he": "MENA",
    "ru": "CIS", "uk": "CIS",
    "de": "Europe", "fr": "Europe", "it": "Europe", "pl": "Europe", "tr": "Europe",
    "cs": "Europe", "hu": "Europe", "el": "Europe", "fi": "Europe", "sv": "Europe",
    "da": "Europe", "no": "Europe", "nl": "Europe", "ro": "Europe", "bg": "Europe",
    "sk": "Europe", "hr": "Europe",
    "pt": "Latin America",
}

conn = get_connection()
lang_mix_rows = conn.execute(
    "SELECT title_id, language_code, SUM(viewer_count) AS viewer_time "
    "FROM language_mix_snapshots WHERE title_id IN ({}) GROUP BY title_id, language_code".format(
        ",".join("?" * len(all_title_ids))
    ),
    all_title_ids,
).fetchall()
conn.close()

lang_mix_df = pd.DataFrame(lang_mix_rows, columns=["title_id", "language_code", "viewer_time"])
lang_mix_df["region"] = lang_mix_df["language_code"].map(LANGUAGE_TO_REGION)

title_totals = lang_mix_df.groupby("title_id")["viewer_time"].sum()
coverage_by_id = (
    lang_mix_df[lang_mix_df["region"].notna()].groupby("title_id")["viewer_time"].sum() / title_totals * 100
).reindex(all_title_ids).fillna(0.0)

dominant_audience_region_by_id = {}
for title_id in all_title_ids:
    sub = lang_mix_df[(lang_mix_df["title_id"] == title_id) & lang_mix_df["region"].notna()]
    if len(sub) == 0 or sub["viewer_time"].sum() == 0:
        dominant_audience_region_by_id[title_id] = "Unclassified (no mapped audience-language data)"
    else:
        dominant_audience_region_by_id[title_id] = sub.groupby("region")["viewer_time"].sum().idxmax()

pd.DataFrame({
    "title": [display_name_by_id[t] for t in all_title_ids],
    "dominant_audience_region": [dominant_audience_region_by_id[t] for t in all_title_ids],
    "pct_viewer_time_mapped": coverage_by_id.round(1).values,
}).sort_values("pct_viewer_time_mapped", ascending=False)

,title,dominant_audience_region,pct_viewer_time_mapped
10,Mobile Legends: Bang Bang,CIS,93.3
4,Dota 2,CIS,82.3
3,Counter-Strike 2,CIS,77.6
18,Street Fighter 6,Asia-Pacific,71.4
14,PUBG Mobile,CIS,69.7
13,PUBG: BATTLEGROUNDS,CIS,67.6
22,League of Legends: Wild Rift,Latin America,66.9
1,Apex Legends,Asia-Pacific,56.3
2,Brawl Stars,Europe,54.7
19,Teamfight Tactics,Europe,40.2


In [10]:
# Publisher, measured directly at the pair level -- identical test to
# creator_crossover_community_detection.ipynb's updated version, not
# re-run independently since it doesn't depend on which community-
# detection algorithm was used (it's a property of pair_df, computed
# before any partition exists).
pair_df["publisher_a"] = pair_df["a"].map(publisher_by_id)
pair_df["publisher_b"] = pair_df["b"].map(publisher_by_id)
pair_df["same_publisher"] = pair_df["publisher_a"] == pair_df["publisher_b"]

same_pub = pair_df[pair_df["same_publisher"]]
diff_pub = pair_df[~pair_df["same_publisher"]]

print(f"{len(same_pub)} same-publisher pairs (of {len(pair_df)} total)")
print(f"  mean enrichment_ratio:   {same_pub['enrichment'].mean():.2f}")
print(f"  {(same_pub['enrichment'] > 1.0).sum()} of {len(same_pub)} clear the chance baseline")
print(f"{len(diff_pub)} different-publisher pairs")
print(f"  mean enrichment_ratio:   {diff_pub['enrichment'].mean():.2f}")
print(f"  {(diff_pub['enrichment'] > 1.0).sum()} of {len(diff_pub)} clear the chance baseline")

11 same-publisher pairs (of 253 total)
  mean enrichment_ratio:   0.89
  3 of 11 clear the chance baseline
242 different-publisher pairs
  mean enrichment_ratio:   0.34
  14 of 242 clear the chance baseline


In [11]:
# Cross-tab + purity report, identical method to the Louvain notebook,
# applied to Leiden's raw-weight communities.
community_leiden_by_id = {t: i for i, comm in enumerate(communities_leiden_raw) for t in comm}

crosstab_df = pd.DataFrame({
    "title": [display_name_by_id[t] for t in all_title_ids],
    "community_leiden": [community_leiden_by_id[t] for t in all_title_ids],
    "genre": [genre_by_id[t] for t in all_title_ids],
    "platform": [platform_by_id[t] for t in all_title_ids],
    "publisher": [publisher_by_id[t] for t in all_title_ids],
    "audience_region": [dominant_audience_region_by_id[t] for t in all_title_ids],
}).sort_values(["community_leiden", "title"]).reset_index(drop=True)

def purity_report(df: pd.DataFrame, community_col: str, vector_cols: list[str]) -> pd.DataFrame:
    rows = []
    for comm_id, sub in df.groupby(community_col):
        if len(sub) < 2:
            continue
        row = {"community": comm_id, "n_titles": len(sub), "titles": ", ".join(sub["title"])}
        for col in vector_cols:
            modal_value = sub[col].value_counts().idxmax()
            modal_share = sub[col].value_counts().iloc[0] / len(sub)
            row[f"{col}_modal"] = modal_value
            row[f"{col}_purity"] = round(modal_share, 2)
        rows.append(row)
    return pd.DataFrame(rows)

display(crosstab_df)
purity_report(crosstab_df, "community_leiden", ["genre", "platform", "publisher", "audience_region"])

,title,community_leiden,genre,platform,publisher,audience_region
0,Apex Legends,0,battle_royale,pc_console,Electronic Arts,Asia-Pacific
1,Fortnite,0,battle_royale,pc_console,Epic Games,Europe
2,League of Legends,0,moba,pc,Riot Games,Europe
3,Overwatch,0,hero_shooter,pc,Blizzard Entertainment,Asia-Pacific
4,Rainbow Six Siege,0,tac_fps,pc,Ubisoft,Europe
5,Rocket League,0,sports,pc,Epic Games,Europe
6,Teamfight Tactics,0,autobattler,mobile,Riot Games,Europe
7,VALORANT,0,tac_fps,pc,Riot Games,Europe
8,Age of Empires II,1,rts,pc,Xbox Game Studios,Europe
9,Counter-Strike 2,1,tac_fps,pc,Valve,CIS


,community,n_titles,titles,genre_modal,genre_purity,platform_modal,platform_purity,publisher_modal,publisher_purity,audience_region_modal,audience_region_purity
0,0,8,"Apex Legends, Fortnite, League of Legends, Ove...",battle_royale,0.25,pc,0.62,Riot Games,0.38,Europe,0.75
1,1,6,"Age of Empires II, Counter-Strike 2, Dota 2, H...",rts,0.33,pc,0.67,Valve,0.33,CIS,0.67
2,2,5,"Brawl Stars, Free Fire, League of Legends: Wil...",battle_royale,0.40,mobile,1.00,Supercell,0.20,Latin America,0.40
3,3,4,"Guilty Gear -Strive-, Mortal Kombat 1, Street ...",fighting,1.00,console,0.75,Arc System Works,0.25,Asia-Pacific,0.50


### Read

**Before anything else**: `creator_crossover_null_model_test.ipynb` has since run a configuration-model null test on the raw-weight graph, re-run itself on 2026-10-02. Since Leiden and Louvain find the identical raw-weight partition (below), that test applies to both: Communities 1-3 are statistically real (z=2.74, 10.80, 14.97), **Community 0 is not** (44.6th percentile of its own null distribution, z=-0.27) — it's a residual grouping, not a fourth community. Read anything below about Community 0 accordingly.

**Updated 2026-10-02 against current data** (296,583 channels, up from 233,540) — see `creator_crossover_community_detection.ipynb`'s own Read for the full account of what moved and why on the raw-weight graph; only the Leiden-specific comparisons are re-derived here. One of those comparisons reversed outright — see below.

**Algorithm choice still doesn't matter on the raw-weight graph — Leiden and Louvain find the identical partition.** Adjusted Rand Index = **1.000**, weighted modularity matches exactly (0.439 both ways). The fighting-game (genre) and mobile (platform) clusters are not an artifact of which algorithm ran.

**Algorithm choice no longer matters on the rank-weighted graph either — this directly reverses the previous run's finding.** The previous run found a genuine algorithm-dependent divergence: Louvain kept the full 5-title mobile cluster together under rank weights, Leiden split Wild Rift off into a different community (4 of 5 mobile titles together, not 5 of 5). **With five weeks of data instead of two, that divergence is gone.** Leiden's rank-weight run now finds the same **3** communities as Louvain's rank-weight run, with identical membership title-for-title: {Apex Legends, Fortnite, Guilty Gear -Strive-, Mortal Kombat 1, Overwatch, Rainbow Six Siege, Rocket League, Street Fighter 6, Tekken 8, VALORANT} (10), {Age of Empires II, Counter-Strike 2, Dota 2, Hearthstone, League of Legends, PUBG: BATTLEGROUNDS, StarCraft II, Teamfight Tactics} (8), and {Brawl Stars, Free Fire, League of Legends: Wild Rift, Mobile Legends: Bang Bang, PUBG Mobile} (5) — the mobile cluster, fully intact under both algorithms now. **Read this as the two-week window being too short to pin down a boundary case, not as the previous report having been wrong at the time** — Wild Rift's defection was a real artifact of where Leiden happened to draw a boundary on thin data, and it resolved once there was more data to draw the boundary with.

**Leiden's own raw-vs-rank Adjusted Rand Index is now 0.535** (was 0.463; Louvain's own is also 0.535, up from 0.407 — identical now, consistent with the two algorithms' rank partitions having converged completely, not just gotten closer).

**A real bug caught along the way, still worth flagging for anyone reusing this pattern**: `leidenalg`'s `partition.modularity` attribute silently returns the *unweighted* modularity of the found partition, even when `weights="weight"` is correctly passed to `find_partition`. Use the explicit weighted call (`Gig.modularity(membership, weights=...)`), not the convenience attribute.

**Net effect on the original question, current as of this run**: genre (fighting) and platform (mobile) are algorithm-independent findings on the graph that actually matters (raw weights), confirmed statistically real. Publisher and audience-region are algorithm-independent for the same reason (same partition). **Robustness to raw-vs-rank weighting improved for both algorithms with more data, and the two algorithms now fully agree with each other on the rank-weighted graph too** — the "new wrinkle" the previous run surfaced (Leiden and Louvain disagreeing specifically on the rank-weighted mobile cluster) did not hold up; more data resolved it rather than confirming it as a real algorithm-dependent effect.

---

**Addendum, 2026-09-20 (unchanged by this data refresh)**: `region_coverage_artifact_test.ipynb` found that the audience-region variable's "Europe" reads are a coverage artifact (Europe is mapped from 17 languages against Latin America's 1 and CIS's 2, so excluding English or Spanish hands Europe the "what's left" contest structurally). Since this notebook's partition is identical to the Louvain notebook's, the same correction applies here without re-deriving it: Community 0's and Community 3's region purity claims should be retired; Community 1's CIS lean survives on Counter-Strike and Dota 2 specifically. See the other notebook's own addendum for the full statistics.